In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
from sklearn.linear_model import LogisticRegression
from numpy.random import poisson
import inspect
import os

fifa_ranking_2022 = pd.read_csv('fifa_ranking_2022-10-06.csv')

fifa_ranking_2026 = pd.read_csv('fifa_ranking_2026-06-08.csv')

matches = pd.read_csv('matches_1930_2022.csv')

schedule_2026 = pd.read_csv('schedule_2026.csv')

world_cup = pd.read_csv('world_cup.csv')

print("Successfully loaded all datasets.")

Successfully loaded all datasets.


In [39]:
matches["Date"] = pd.to_datetime(matches["Date"])

matches = (
    matches
    .sort_values("Date")
    .reset_index(drop=True)
) # Sort matches by date

print(matches[["Date", "home_team", "away_team"]].head())

        Date      home_team away_team
0 1930-07-13         France    Mexico
1 1930-07-13  United States   Belgium
2 1930-07-14     Yugoslavia    Brazil
3 1930-07-14        Romania      Peru
4 1930-07-15      Argentina    France


In [40]:
team_mapping = {
    "Czechia": "Czech Republic",
    "United States" : "USA",
    "Bosnia-Herzegovina" : "Bosnia and Herzegovina",
    "Congo DR" : "Congo",
    "Curaçao": "Curacao",
}

schedule_2026['home_team'] = schedule_2026["home_team"].replace(team_mapping)
schedule_2026['away_team'] = schedule_2026["away_team"].replace(team_mapping)
matches['home_team'] = matches["home_team"].replace(team_mapping)
matches['away_team'] = matches["away_team"].replace(team_mapping)

In [41]:
def clean_team(x):
    return str(x).strip()

schedule_2026["home_team"] = schedule_2026["home_team"].apply(clean_team)
schedule_2026["away_team"] = schedule_2026["away_team"].apply(clean_team)

In [42]:
INITIAL_ELO = 1500 # Initial Elo rating for all teams

elo = defaultdict(lambda: INITIAL_ELO) # Default dictionary to store Elo ratings for each team

matches["home_elo"] = 0.0
matches["away_elo"] = 0.0
matches["elo_diff"] = 0.0

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

K = 30

def update_elo(home_rating, away_rating, home_goals, away_goals):

    expected_home = expected_score(home_rating, away_rating)
    expected_away = expected_score(away_rating, home_rating)

    if home_goals > away_goals:
        actual_home = 1
        actual_away = 0

    elif home_goals < away_goals:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    new_home = home_rating + K * (actual_home - expected_home)
    new_away = away_rating + K * (actual_away - expected_away)

    return new_home, new_away


for idx, row in matches.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_rating = elo[home]
    away_rating = elo[away]

    # Save ratings BEFORE the match
    matches.loc[idx, "home_elo"] = home_rating
    matches.loc[idx, "away_elo"] = away_rating
    matches.loc[idx, "elo_diff"] = home_rating - away_rating

    # Update ratings AFTER the match
    new_home, new_away = update_elo(
        home_rating,
        away_rating,
        row["home_score"],
        row["away_score"]
    )

    elo[home] = new_home
    elo[away] = new_away


elo_table = pd.DataFrame(
    {
        "team": list(elo.keys()),
        "elo": list(elo.values())
    }
)

elo_table = elo_table.sort_values(
    "elo",
    ascending=False
)

print(elo_table.head(20).to_string(index=False))

        team         elo
 Netherlands 1701.686750
      Brazil 1692.357409
      France 1691.320681
     Germany 1673.262273
   Argentina 1658.758350
West Germany 1650.090500
       Italy 1619.111830
       Spain 1594.447707
     Belgium 1582.822762
     England 1578.102002
     Croatia 1571.805689
    Portugal 1548.063310
      Sweden 1545.737935
Soviet Union 1544.993080
     Uruguay 1543.762623
  Yugoslavia 1534.388048
     Türkiye 1532.566087
    Colombia 1524.315216
     Senegal 1524.087745
     Romania 1524.030608


In [43]:
matches["home_last5_win_rate"] = 0.0
matches["away_last5_win_rate"] = 0.0

matches["home_last5_draw_rate"] = 0.0
matches["away_last5_draw_rate"] = 0.0

matches["home_last5_loss_rate"] = 0.0
matches["away_last5_loss_rate"] = 0.0

matches["home_last5_goals_for"] = 0.0
matches["away_last5_goals_for"] = 0.0

matches["home_last5_goals_against"] = 0.0
matches["away_last5_goals_against"] = 0.0

def calculate_form(history):

    if len(history) == 0:
        return {
            "win_rate": 0,
            "draw_rate": 0,
            "loss_rate": 0,
            "goals_for": 0,
            "goals_against": 0
        }

    history = history[-5:]

    return {
        "win_rate": np.mean([h["win"] for h in history]),
        "draw_rate": np.mean([h["draw"] for h in history]),
        "loss_rate": np.mean([h["loss"] for h in history]),
        "goals_for": np.mean([h["goals_for"] for h in history]),
        "goals_against": np.mean([h["goals_against"] for h in history])
    }

In [44]:
team_history = defaultdict(list)

for idx, row in matches.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_form = calculate_form(team_history[home])
    away_form = calculate_form(team_history[away])

    matches.loc[idx, "home_last5_win_rate"] = home_form["win_rate"]
    matches.loc[idx, "away_last5_win_rate"] = away_form["win_rate"]

    matches.loc[idx, "home_last5_draw_rate"] = home_form["draw_rate"]
    matches.loc[idx, "away_last5_draw_rate"] = away_form["draw_rate"]

    matches.loc[idx, "home_last5_loss_rate"] = home_form["loss_rate"]
    matches.loc[idx, "away_last5_loss_rate"] = away_form["loss_rate"]

    matches.loc[idx, "home_last5_goals_for"] = home_form["goals_for"]
    matches.loc[idx, "away_last5_goals_for"] = away_form["goals_for"]

    matches.loc[idx, "home_last5_goals_against"] = home_form["goals_against"]
    matches.loc[idx, "away_last5_goals_against"] = away_form["goals_against"]

    home_win = int(row["home_score"] > row["away_score"])
    home_draw = int(row["home_score"] == row["away_score"])
    home_loss = int(row["home_score"] < row["away_score"])

    away_win = home_loss
    away_draw = home_draw
    away_loss = home_win

    team_history[home].append({

        "win": home_win,
        "draw": home_draw,
        "loss": home_loss,

        "goals_for": row["home_score"],
        "goals_against": row["away_score"]

    })

    team_history[away].append({

        "win": away_win,
        "draw": away_draw,
        "loss": away_loss,

        "goals_for": row["away_score"],
        "goals_against": row["home_score"]

    })

matches["form_diff"] = (
    matches["home_last5_win_rate"]
    - matches["away_last5_win_rate"]
)

matches["attack_diff"] = (
    matches["home_last5_goals_for"]
    - matches["away_last5_goals_for"]
)

matches["defense_diff"] = (
    matches["away_last5_goals_against"]
    - matches["home_last5_goals_against"]
)

matches["draw_rate_diff"] = (
    matches["home_last5_draw_rate"]
    - matches["away_last5_draw_rate"]
)

print(
    matches[
        [
            "home_team",
            "away_team",
            "home_last5_win_rate",
            "away_last5_win_rate",
            "home_last5_goals_for",
            "away_last5_goals_for"
        ]
    ].head(20).to_string(index=False)
)

     home_team  away_team  home_last5_win_rate  away_last5_win_rate  home_last5_goals_for  away_last5_goals_for
        France     Mexico                  0.0                  0.0              0.000000                   0.0
           USA    Belgium                  0.0                  0.0              0.000000                   0.0
    Yugoslavia     Brazil                  0.0                  0.0              0.000000                   0.0
       Romania       Peru                  0.0                  0.0              0.000000                   0.0
     Argentina     France                  0.0                  1.0              0.000000                   4.0
         Chile     Mexico                  0.0                  0.0              0.000000                   1.0
    Yugoslavia    Bolivia                  1.0                  0.0              2.000000                   0.0
           USA   Paraguay                  1.0                  0.0              3.000000               

In [45]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return "Home" # Home team win
    elif row['home_score'] < row['away_score']:
        return "Away" # Away team win
    else:
        return "Draw" # Draw

matches['result'] = matches.apply(get_result, axis=1)

features = [

    "home_elo",
    "away_elo",
    "elo_diff",

    "home_last5_win_rate",
    "away_last5_win_rate",

    "home_last5_draw_rate",
    "away_last5_draw_rate",

    "home_last5_loss_rate",
    "away_last5_loss_rate",

    "home_last5_goals_for",
    "away_last5_goals_for",

    "home_last5_goals_against",
    "away_last5_goals_against",

    "form_diff",
    "attack_diff",
    "defense_diff",
    "draw_rate_diff"
]

scaler = StandardScaler()

X = matches[features]

y = matches["result"]

split = int(len(matches) * 0.8)

train = matches.iloc[:split]
test = matches.iloc[split:]

X_train = train[features]
X_test = test[features]

y_train = train["result"]
y_test = test["result"]

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

model = LogisticRegression(

    max_iter=5000,

    class_weight="balanced",

    random_state=42
)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

print(classification_report(
    y_test,
    prediction
))

              precision    recall  f1-score   support

        Away       0.49      0.52      0.50        69
        Draw       0.20      0.29      0.24        41
        Home       0.55      0.40      0.46        83

    accuracy                           0.42       193
   macro avg       0.41      0.40      0.40       193
weighted avg       0.45      0.42      0.43       193



In [46]:
forest = RandomForestClassifier(

    n_estimators=500,

    max_depth=10,

    min_samples_leaf=5,

    class_weight="balanced_subsample",

    random_state=42
)

forest.fit(train[features], y_train)

forest_prediction = forest.predict(
    test[features]
)

print(
    classification_report(
        y_test,
        forest_prediction
    )
)

              precision    recall  f1-score   support

        Away       0.47      0.48      0.47        69
        Draw       0.19      0.12      0.15        41
        Home       0.57      0.66      0.61        83

    accuracy                           0.48       193
   macro avg       0.41      0.42      0.41       193
weighted avg       0.45      0.48      0.46       193



In [47]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance": forest.feature_importances_

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.to_string(index=False))

                 Feature  Importance
                elo_diff    0.122703
                away_elo    0.119811
                home_elo    0.102333
             attack_diff    0.080142
    home_last5_goals_for    0.071454
    away_last5_goals_for    0.068105
            defense_diff    0.063908
away_last5_goals_against    0.062399
home_last5_goals_against    0.060883
               form_diff    0.042024
          draw_rate_diff    0.033402
    away_last5_loss_rate    0.031283
    home_last5_loss_rate    0.030394
     home_last5_win_rate    0.028139
     away_last5_win_rate    0.028128
    away_last5_draw_rate    0.027728
    home_last5_draw_rate    0.027164


In [48]:
elo_latest = pd.DataFrame({

    "team": elo.keys(),

    "elo": elo.values()

})

elo_latest = elo_latest.sort_values(

    "elo",

    ascending=False
)

team_features = []

for team, history in team_history.items():

    stats = calculate_form(history)

    team_features.append({

        "team": team,

        "elo": elo[team],

        "win_rate": stats["win_rate"],

        "draw_rate": stats["draw_rate"],

        "loss_rate": stats["loss_rate"],

        "goals_for": stats["goals_for"],

        "goals_against": stats["goals_against"]

    })

team_features = pd.DataFrame(team_features)

print(team_features.head().to_string(index=False))

      team         elo  win_rate  draw_rate  loss_rate  goals_for  goals_against
    France 1691.320681       0.6        0.2        0.2        2.0            1.2
    Mexico 1520.936771       0.2        0.2        0.6        0.4            1.6
       USA 1452.157406       0.2        0.4        0.4        0.8            1.2
   Belgium 1582.822762       0.4        0.2        0.4        0.6            0.6
Yugoslavia 1534.388048       0.6        0.2        0.2        1.6            1.2


In [49]:
team_dict = {}

for _, row in team_features.iterrows():  # type: ignore

    team_dict[row["team"]] = row.to_dict()



In [50]:
home = team_dict["Brazil"]

away = team_dict["France"]

feature_row = {

    "home_elo": home["elo"],
    "away_elo": away["elo"],
    "elo_diff": home["elo"] - away["elo"],

    "home_last5_win_rate": home["win_rate"],
    "away_last5_win_rate": away["win_rate"],

    "home_last5_draw_rate": home["draw_rate"],
    "away_last5_draw_rate": away["draw_rate"],

    "home_last5_loss_rate": home["loss_rate"],
    "away_last5_loss_rate": away["loss_rate"],

    "home_last5_goals_for": home["goals_for"],
    "away_last5_goals_for": away["goals_for"],

    "home_last5_goals_against": home["goals_against"],
    "away_last5_goals_against": away["goals_against"],

    "form_diff":
        home["win_rate"] - away["win_rate"],

    "attack_diff":
        home["goals_for"] - away["goals_for"],

    "defense_diff":
        away["goals_against"] - home["goals_against"],

    "draw_rate_diff":
        home["draw_rate"] - away["draw_rate"]
}

feature_df = pd.DataFrame([feature_row])

feature_scaled = scaler.transform(feature_df)

prediction = model.predict(feature_scaled)

prob = model.predict_proba(feature_scaled)

print(prediction)

print(prob)

['Away']
[[0.43956409 0.25251119 0.30792473]]


In [51]:
import unicodedata

default_team = {
    "elo": 1500,
    "win_rate": 0.33,
    "draw_rate": 0.33,
    "loss_rate": 0.34,
    "goals_for": 1.0,
    "goals_against": 1.0
}

def normalize(name):
    return unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")

def build_match_features(home_team, away_team):

    home = team_dict.get(home_team, default_team)
    away = team_dict.get(away_team, default_team)

    return pd.DataFrame([{
        "home_elo": home["elo"],
        "away_elo": away["elo"],
        "elo_diff": home["elo"] - away["elo"],

        "home_last5_win_rate": home["win_rate"],
        "away_last5_win_rate": away["win_rate"],

        "home_last5_draw_rate": home["draw_rate"],
        "away_last5_draw_rate": away["draw_rate"],

        "home_last5_loss_rate": home["loss_rate"],
        "away_last5_loss_rate": away["loss_rate"],

        "home_last5_goals_for": home["goals_for"],
        "away_last5_goals_for": away["goals_for"],

        "home_last5_goals_against": home["goals_against"],
        "away_last5_goals_against": away["goals_against"],

        "form_diff": home["win_rate"] - away["win_rate"],
        "attack_diff": home["goals_for"] - away["goals_for"],
        "defense_diff": away["goals_against"] - home["goals_against"],
        "draw_rate_diff": home["draw_rate"] - away["draw_rate"]
    }])

X = build_match_features(
    "Brazil",
    "France"
)

prediction = model.predict(
    scaler.transform(X)
)

probability = model.predict_proba(
    scaler.transform(X)
)

print(prediction)
print(probability)

['Away']
[[0.43956409 0.25251119 0.30792473]]


In [52]:
results = []

for _, row in schedule_2026.iterrows():

    X = build_match_features(
        row["home_team"],
        row["away_team"]
    )

    X_scaled = scaler.transform(X)

    pred = model.predict(X_scaled)[0]

    prob = model.predict_proba(X_scaled)[0]

    results.append({

        "home_team": row["home_team"],

        "away_team": row["away_team"],

        "prediction": pred,

        "Away Probability": prob[0],

        "Draw Probability": prob[1],

        "Home Probability": prob[2]

    })

predictions = pd.DataFrame(results)

print(predictions.head(50).to_string(index=False))

             home_team              away_team prediction  Away Probability  Draw Probability  Home Probability
                Mexico           South Africa       Draw          0.354925          0.391698          0.253378
        Korea Republic         Czech Republic       Draw          0.387461          0.389098          0.223441
                Canada Bosnia and Herzegovina       Away          0.443777          0.397131          0.159092
                   USA               Paraguay       Away          0.517951          0.362388          0.119661
                 Qatar            Switzerland       Draw          0.313848          0.456482          0.229671
                Brazil                Morocco       Home          0.174125          0.408693          0.417182
                 Haiti               Scotland       Draw          0.264870          0.446625          0.288505
             Australia                Türkiye       Away          0.431015          0.382652          0.186333
 

In [53]:
def get_points(result):
    if result == "Home":
        return (3, 0)
    elif result == "Away":
        return (0, 3)
    else:
        return (1, 1)
    
groups_tables = {
    "A": {"Mexico": {}, "South Africa": {}, "South Korea": {}, "Czechia": {}},
    "B": {"Canada": {}, "Bosnia and Herzegovina": {}, "Qatar": {}, "Switzerland": {}},
    "C": {"Brazil": {}, "Morocco": {}, "Haiti": {}, "Scotland": {}},
    "D": {"United States": {}, "Paraguay": {}, "Australia": {}, "Turkey": {}},
    "E": {"Germany": {}, "Curacao": {}, "Ivory Coast": {}, "Ecuador": {}},
    "F": {"Netherlands": {}, "Japan": {}, "Sweden": {}, "Tunisia": {}},
    "G": {"Belgium": {}, "Egypt": {}, "Iran": {}, "New Zealand": {}},
    "H": {"Spain": {}, "Cape Verde": {}, "Saudi Arabia": {}, "Uruguay": {}},
    "I": {"France": {}, "Senegal": {}, "Iraq": {}, "Norway": {}},
    "J": {"Argentina": {}, "Algeria": {}, "Austria": {}, "Jordan": {}},
    "K": {"Portugal": {}, "DR Congo": {}, "Uzbekistan": {}, "Colombia": {}},
    "L": {"England": {}, "Croatia": {}, "Ghana": {}, "Panama": {}}
}

group_map = {}

for g, teams in groups_tables.items():
    for t in teams.keys():
        group_map[clean_team(t)] = g

schedule_2026["home_team"] = schedule_2026["home_team"].apply(clean_team)
schedule_2026["away_team"] = schedule_2026["away_team"].apply(clean_team)

schedule_2026["group"] = schedule_2026["home_team"].map(group_map)

group_fixtures = {}

for g in groups_tables.keys():

    group_fixtures[g] = (
        schedule_2026[schedule_2026["group"] == g]
        [["home_team", "away_team"]]
        .reset_index(drop=True)
    )

In [54]:
def simulate_goals(home_attack, away_defense, away_attack, home_defense):

    home_xg = max(0.2, home_attack - away_defense)
    away_xg = max(0.2, away_attack - home_defense)

    home_goals = poisson(home_xg)
    away_goals = poisson(away_xg)

    return home_goals, away_goals

def update_points(table, home, away, home_goals, away_goals):

    if home_goals > away_goals:
        table[home]["pts"] += 3
    elif home_goals < away_goals:
        table[away]["pts"] += 3
    else:
        table[home]["pts"] += 1
        table[away]["pts"] += 1

    table[home]["gf"] += home_goals
    table[away]["gf"] += away_goals

    table[home]["gd"] += home_goals - away_goals
    table[away]["gd"] += away_goals - home_goals

def simulate_match(home, away, model, scaler):

    X = build_match_features(home, away)
    X = scaler.transform(X)

    prob = model.predict_proba(X)[0]

    # Convert ML probability → expected attacking strength
    home_xg = 1.2 + (prob[2] - prob[0]) * 1.5
    away_xg = 1.2 + (prob[0] - prob[2]) * 1.5

    home_xg = max(0.2, home_xg)
    away_xg = max(0.2, away_xg)

    # Simulate goals using Poisson
    home_goals = poisson(home_xg)
    away_goals = poisson(away_xg)

    return home, away, home_goals, away_goals

def simulate_group(fixtures, model, scaler, team_dict):

    teams = pd.concat([
        fixtures["home_team"],
        fixtures["away_team"]
    ]).unique()

    table = {
        team: {"pts": 0, "gd": 0, "gf": 0}
        for team in teams
    }

    for _, row in fixtures.iterrows():

        home, away, hg, ag = simulate_match(
            row["home_team"],
            row["away_team"],
            model,
            scaler
        )

        update_points(table, home, away, hg, ag)

    return table

def rank_group(table):

    return sorted(
        table.items(),
        key=lambda x: (
            x[1]["pts"],
            x[1]["gd"],
            x[1]["gf"]
        ),
        reverse=True
    )

def simulate_groups(groups_tables, group_fixtures, model, scaler):

    knockout_teams = []

    for group_name in groups_tables.keys():

        fixtures = group_fixtures[group_name]

        table = simulate_group(
            fixtures,
            model,
            scaler,
            team_dict
        )

        ranking = rank_group(table)

        top2 = [ranking[0][0], ranking[1][0]]

        knockout_teams.extend(top2)

    return knockout_teams

def simulate_knockout_match(home, away, model, scaler):

    X = build_match_features(home, away)
    X = scaler.transform(X)

    prob = model.predict_proba(X)[0]

    home, away, hg, ag = simulate_match(home, away, model, scaler)

    if hg > ag:
        return home
    elif ag > hg:
        return away
    else:
    # extra time + randomness
        return np.random.choice([home, away])

    if result == "Home":
        return home
    else:
        return away
    
def simulate_knockout(teams, model, scaler):

    # Round of 16
    qf = []

    for i in range(0, len(teams), 2):

        winner = simulate_knockout_match(
            teams[i],
            teams[i+1],
            model,
            scaler
        )

        qf.append(winner)

    # Quarterfinals
    sf = []

    for i in range(0, len(qf), 2):

        winner = simulate_knockout_match(
            qf[i],
            qf[i+1],
            model,
            scaler
        )

        sf.append(winner)

    # Semifinals
    final = []

    for i in range(0, len(sf), 2):

        winner = simulate_knockout_match(
            sf[i],
            sf[i+1],
            model,
            scaler
        )

        final.append(winner)

    # Final
    champion = simulate_knockout_match(
        final[0],
        final[1],
        model,
        scaler
    )

    return champion

In [55]:
def simulate_world_cup(groups_tables, group_fixtures, model, scaler):

    # GROUP STAGE
    knockout_teams = simulate_groups(
        groups_tables,
        group_fixtures,
        model,
        scaler
    )

    # KNOCKOUT STAGE
    champion = simulate_knockout(
        knockout_teams,
        model,
        scaler
    )

    return champion

teams_in_knockout = simulate_world_cup(groups_tables, group_fixtures, model, scaler)
print(teams_in_knockout)

Brazil


In [56]:
def run_monte_carlo(n_simulations, groups_tables, group_fixtures, model, scaler):

    results = Counter()

    for _ in range(n_simulations):

        winner = simulate_world_cup(
            groups_tables,
            group_fixtures,
            model,
            scaler
        )

        results[winner] += 1

    # convert to probabilities
    for team in results:
        results[team] /= n_simulations

    return results

teams = [clean_team(t) for t in teams]

results = run_monte_carlo(
    n_simulations=1000,
    groups_tables=groups_tables,
    group_fixtures=group_fixtures,
    model=model,
    scaler=scaler
)

print(results)

Counter({np.str_('Netherlands'): 0.222, np.str_('Spain'): 0.216, np.str_('Brazil'): 0.191, np.str_('Germany'): 0.103, np.str_('Belgium'): 0.058, np.str_('Türkiye'): 0.04, 'Uruguay': 0.027, np.str_('Cape Verde'): 0.027, np.str_('Sweden'): 0.019, 'Paraguay': 0.011, np.str_('Australia'): 0.01, np.str_('Bosnia and Herzegovina'): 0.01, np.str_('Korea Republic'): 0.007, np.str_('Curacao'): 0.007, np.str_('Mexico'): 0.007, np.str_('South Africa'): 0.006, np.str_('Morocco'): 0.006, np.str_('Switzerland'): 0.006, np.str_('Egypt'): 0.005, np.str_("Côte d'Ivoire"): 0.005, np.str_('Qatar'): 0.004, np.str_('IR Iran'): 0.004, 'Saudi Arabia': 0.003, np.str_('Ecuador'): 0.002, np.str_('New Zealand'): 0.002, np.str_('Scotland'): 0.001, np.str_('Japan'): 0.001})
